### NEWTON'S ALGORITHM

In [2]:

def dot(a, b):
    return sum(ai * bi for ai, bi in zip(a, b))

def norm(a):
    return dot(a, a) ** 0.5

def vec_add(a, b):
    return [ai + bi for ai, bi in zip(a, b)]

def vec_scale(c, a):
    return [c * ai for ai in a]


# Solve A d = b (Gaussian elimination, partial pivoting)
def solve(A, b):
    n = len(b)
    M = [row[:] + [b[i]] for i, row in enumerate(A)]     # augmented matrix
    for col in range(n):
        piv = max(range(col, n), key=lambda r: abs(M[r][col]))
        if abs(M[piv][col]) < 1e-14:
            return None                                   # singular matrix
        M[col], M[piv] = M[piv], M[col]
        for r in range(col + 1, n):
            factor = M[r][col] / M[col][col]
            for c in range(col, n + 1):
                M[r][c] -= factor * M[col][c]
    d = [0.0] * n                                         # back substitution
    for i in range(n - 1, -1, -1):
        s = sum(M[i][j] * d[j] for j in range(i + 1, n))
        d[i] = (M[i][n] - s) / M[i][i]
    return d


# Line search: Armijo backtracking
def backtracking(f, x, fx, g, p, c1=1e-4, rho=0.5, max_halvings=50):
    t = 1.0
    slope = dot(g, p)
    for _ in range(max_halvings):
        if f(vec_add(x, vec_scale(t, p))) <= fx + c1 * t * slope:
            break
        t *= rho
    return t


# Newton's method
def newton(f, grad, hess, x0, tol=1e-8, max_iter=100):
    x = list(x0)
    k = 0
    for k in range(max_iter):
        g = grad(x)
        if norm(g) < tol:
            break

        # Newton direction: solve  H p = -g   (no matrix inverse needed)
        p = solve(hess(x), vec_scale(-1.0, g))

        # Safeguard: if H is singular or p is not a descent direction,
        # fall back to steepest descent for this iteration
        if p is None or dot(g, p) >= 0:
            p = vec_scale(-1.0, g)

        t = backtracking(f, x, f(x), g, p)
        x = vec_add(x, vec_scale(t, p))

    return x, f(x), k

# Test: Rosenbrock, minimum at (1, 1)
def f(x):
    return 100 * (x[1] - x[0] ** 2) ** 2 + (1 - x[0]) ** 2

def grad(x):
    return [-400 * x[0] * (x[1] - x[0] ** 2) - 2 * (1 - x[0]),
            200 * (x[1] - x[0] ** 2)]

def hess(x):
    return [[1200 * x[0] ** 2 - 400 * x[1] + 2, -400 * x[0]],
            [-400 * x[0],                        200.0]]

x_opt, f_opt, iters = newton(f, grad, hess, [-1.2, 1.0])
print("x*         =", x_opt)
print("f(x*)      =", f_opt)
print("iterations =", iters)

x*         = [0.9999999999400667, 0.9999999998789006]
f(x*)      = 3.743975643139474e-21
iterations = 21


### QUASI-NEWTON'S ALGORITHM

In [1]:

def dot(a, b):
    return sum(ai * bi for ai, bi in zip(a, b))

def norm(a):
    return dot(a, a) ** 0.5

def vec_add(a, b):
    return [ai + bi for ai, bi in zip(a, b)]

def vec_sub(a, b):
    return [ai - bi for ai, bi in zip(a, b)]

def vec_scale(c, a):
    return [c * ai for ai in a]

def identity(n):
    return [[1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]

def mat_vec(M, v):
    return [dot(row, v) for row in M]

def mat_mul(A, B):
    n, m, p = len(A), len(B), len(B[0])
    return [[sum(A[i][k] * B[k][j] for k in range(m)) for j in range(p)]
            for i in range(n)]

def outer(a, b):
    return [[ai * bj for bj in b] for ai in a]

def mat_add(A, B):
    return [[a + b for a, b in zip(ra, rb)] for ra, rb in zip(A, B)]

def mat_sub(A, B):
    return [[a - b for a, b in zip(ra, rb)] for ra, rb in zip(A, B)]

def mat_scale(c, A):
    return [[c * a for a in row] for row in A]


# ---------- Line search: Armijo backtracking ----------
def backtracking(f, x, fx, g, p, c1=1e-4, rho=0.5, max_halvings=50):
    t = 1.0
    slope = dot(g, p)
    for _ in range(max_halvings):
        if f(vec_add(x, vec_scale(t, p))) <= fx + c1 * t * slope:
            break
        t *= rho
    return t


#BFGS
def bfgs(f, grad, x0, tol=1e-8, max_iter=1000):
    x = list(x0)
    n = len(x)
    I = identity(n)
    H = identity(n)              # approximation of the INVERSE Hessian
    fx, g = f(x), grad(x)
    k = 0

    for k in range(max_iter):
        if norm(g) < tol:
            break

        p = vec_scale(-1.0, mat_vec(H, g))       # p = -H g
        if dot(g, p) >= 0:                        # not a descent direction
            H = identity(n)
            p = vec_scale(-1.0, g)

        t = backtracking(f, x, fx, g, p)
        s = vec_scale(t, p)                       # s = x_new - x
        x_new = vec_add(x, s)
        g_new = grad(x_new)
        y = vec_sub(g_new, g)                     # y = g_new - g

        sy = dot(s, y)
        if sy > 1e-10:                            # curvature condition
            if k == 0:                            # scale initial H once
                H = mat_scale(sy / dot(y, y), I)
            rho = 1.0 / sy
            left = mat_sub(I, mat_scale(rho, outer(s, y)))    # I - rho s y^T
            right = mat_sub(I, mat_scale(rho, outer(y, s)))   # I - rho y s^T
            H = mat_add(mat_mul(mat_mul(left, H), right),
                        mat_scale(rho, outer(s, s)))

        x, g, fx = x_new, g_new, f(x_new)

    return x, fx, k


# Test: Rosenbrock, minimum at (1, 1)
def f(x):
    return 100 * (x[1] - x[0] ** 2) ** 2 + (1 - x[0]) ** 2

def grad(x):
    return [-400 * x[0] * (x[1] - x[0] ** 2) - 2 * (1 - x[0]),
            200 * (x[1] - x[0] ** 2)]

x_opt, f_opt, iters = bfgs(f, grad, [-1.2, 1.0])
print("x*         =", x_opt)
print("f(x*)      =", f_opt)
print("iterations =", iters)

x*         = [0.9999999999602242, 0.9999999999245358]
f(x*)      = 3.252890440757666e-21
iterations = 38


### FIXED-STEP GRADIENT

In [4]:
# Basic vector operations (no libraries)
def dot(a, b):
    return sum(ai * bi for ai, bi in zip(a, b))

def norm(a):
    return dot(a, a) ** 0.5

def vec_sub(a, b):
    return [ai - bi for ai, bi in zip(a, b)]

def vec_scale(c, a):
    return [c * ai for ai in a]


# Fixed-step gradient descent 
def gradient_descent(grad, x0, alpha, tol=1e-6, max_iter=100000):
    x = list(x0)
    k = 0
    for k in range(max_iter):
        g = grad(x)
        if norm(g) < tol:                      # stopping criterion: ||grad|| < tol
            break
        x = vec_sub(x, vec_scale(alpha, g))    # x_{k+1} = x_k - alpha * grad f(x_k)
    return x, k


# Test 1: simple quadratic f(x) = (x1 - 3)^2 + 2 (x2 + 1)^2
def f1(x):
    return (x[0] - 3) ** 2 + 2 * (x[1] + 1) ** 2

def grad1(x):
    return [2 * (x[0] - 3), 4 * (x[1] + 1)]

x_opt, iters = gradient_descent(grad1, [0.0, 0.0], alpha=0.1)
print("Quadratic  : x* =", x_opt, "| f(x*) =", f1(x_opt), "| iterations =", iters)


#Test 2: Rosenbrock, minimum at (1, 1)
def f2(x):
    return 100 * (x[1] - x[0] ** 2) ** 2 + (1 - x[0]) ** 2

def grad2(x):
    return [-400 * x[0] * (x[1] - x[0] ** 2) - 2 * (1 - x[0]),
            200 * (x[1] - x[0] ** 2)]

x_opt, iters = gradient_descent(grad2, [-1.2, 1.0], alpha=0.001, tol=1e-6, max_iter=200000)
print("Rosenbrock : x* =", x_opt, "| f(x*) =", f2(x_opt), "| iterations =", iters)

Quadratic  : x* = [2.9999995063486327, -0.9999999999999998] | f(x*) = 2.4369167247591506e-13 | iterations = 70
Rosenbrock : x* = [0.99999888209011, 0.9999977597080407] | f(x*) = 1.2517236788075983e-12 | iterations = 32076


### VARIABLE STEP

In [6]:

def dot(a, b):
    return sum(ai * bi for ai, bi in zip(a, b))

def norm(a):
    return dot(a, a) ** 0.5

def vec_add(a, b):
    return [ai + bi for ai, bi in zip(a, b)]

def vec_sub(a, b):
    return [ai - bi for ai, bi in zip(a, b)]

def vec_scale(c, a):
    return [c * ai for ai in a]

def mat_vec(M, v):
    return [dot(row, v) for row in M]

# Line search: Armijo backtracking
def backtracking(f, x, fx, g, p, c1=1e-4, rho=0.5, max_halvings=60):
    t = 1.0
    slope = dot(g, p)
    for _ in range(max_halvings):
        if f(vec_add(x, vec_scale(t, p))) <= fx + c1 * t * slope:
            break
        t *= rho
    return t

# 1) Gradient descent with VARIABLE step (step chosen at each iteration)

def gradient_variable_step(f, grad, x0, tol=1e-6, max_iter=100000):
    x = list(x0)
    k = 0
    for k in range(max_iter):
        g = grad(x)
        if norm(g) < tol:
            break
        p = vec_scale(-1.0, g)                     # direction: -grad
        t = backtracking(f, x, f(x), g, p)         # step alpha_k found by line search
        x = vec_add(x, vec_scale(t, p))
    return x, k


# Optimal step for a quadratic f(x) = 1/2 x^T A x - b^T x :
#   alpha_k = (g^T g) / (g^T A g)   (exact minimisation along -g)
def gradient_optimal_step_quadratic(A, b, x0, tol=1e-8, max_iter=100000):
    x = list(x0)
    k = 0
    for k in range(max_iter):
        g = vec_sub(mat_vec(A, x), b)              # grad = A x - b
        if norm(g) < tol:
            break
        alpha = dot(g, g) / dot(g, mat_vec(A, g))
        x = vec_sub(x, vec_scale(alpha, g))
    return x, k

### CONJUGATE STEP

In [8]:
# 2) CONJUGATE gradient
# 2a) Linear CG for a quadratic f(x) = 1/2 x^T A x - b^T x   (A symmetric pos. def.)
#     Converges in at most n iterations (exact arithmetic).
def conjugate_gradient_quadratic(A, b, x0, tol=1e-10):
    n = len(b)
    x = list(x0)
    g = vec_sub(mat_vec(A, x), b)                  # g_0 = A x_0 - b
    d = vec_scale(-1.0, g)                         # d_0 = -g_0
    k = 0
    for k in range(n):
        if norm(g) < tol:
            break
        Ad = mat_vec(A, d)
        alpha = dot(g, g) / dot(d, Ad)             # optimal step along d
        x = vec_add(x, vec_scale(alpha, d))
        g_new = vec_add(g, vec_scale(alpha, Ad))   # g_{k+1} = g_k + alpha A d_k
        beta = dot(g_new, g_new) / dot(g, g)       # Fletcher-Reeves
        d = vec_add(vec_scale(-1.0, g_new), vec_scale(beta, d))
        g = g_new
    return x, k

# 2b) Nonlinear CG (Polak-Ribiere+) for any differentiable f
def conjugate_gradient(f, grad, x0, tol=1e-6, max_iter=100000):
    x = list(x0)
    n = len(x)
    g = grad(x)
    d = vec_scale(-1.0, g)
    k = 0
    for k in range(max_iter):
        if norm(g) < tol:
            break
        if dot(g, d) >= 0:                         # not a descent direction: restart
            d = vec_scale(-1.0, g)
        t = backtracking(f, x, f(x), g, d)
        x = vec_add(x, vec_scale(t, d))
        g_new = grad(x)
        beta = max(0.0, dot(g_new, vec_sub(g_new, g)) / dot(g, g))   # PR+
        if (k + 1) % n == 0:                       # periodic restart every n iterations
            beta = 0.0
        d = vec_add(vec_scale(-1.0, g_new), vec_scale(beta, d))
        g = g_new
    return x, k


#TESTS
# Quadratic  f(x) = 1/2 x^T A x - b^T x   with  A = [[4,1],[1,3]],  b = [1,2]
A = [[4.0, 1.0], [1.0, 3.0]]
b = [1.0, 2.0]
fq = lambda x: 0.5 * dot(x, mat_vec(A, x)) - dot(b, x)
gq = lambda x: vec_sub(mat_vec(A, x), b)

x, k = gradient_variable_step(fq, gq, [0.0, 0.0])
print("Variable step (backtracking)     :", x, "| iterations =", k)
x, k = gradient_optimal_step_quadratic(A, b, [0.0, 0.0])
print("Optimal step (quadratic)         :", x, "| iterations =", k)
x, k = conjugate_gradient_quadratic(A, b, [0.0, 0.0])
print("Linear conjugate gradient        :", x, "| iterations =", k + 1)
print("Exact solution of A x = b        : [1/11, 7/11] =", [1 / 11, 7 / 11])

# Rosenbrock, minimum at (1, 1)
def f(x):
    return 100 * (x[1] - x[0] ** 2) ** 2 + (1 - x[0]) ** 2

def grad(x):
    return [-400 * x[0] * (x[1] - x[0] ** 2) - 2 * (1 - x[0]),
            200 * (x[1] - x[0] ** 2)]

print()
x, k = gradient_variable_step(f, grad, [-1.2, 1.0], max_iter=200000)
print("Rosenbrock, variable step        :", x, "| iterations =", k)
x, k = conjugate_gradient(f, grad, [-1.2, 1.0])
print("Rosenbrock, nonlinear CG (PR+)   :", x, "| iterations =", k)

Variable step (backtracking)     : [0.09090924263000488, 0.6363635063171387] | iterations = 13
Optimal step (quadratic)         : [0.0909090906976654, 0.636363634883658] | iterations = 16
Linear conjugate gradient        : [0.09090909090909091, 0.6363636363636364] | iterations = 2
Exact solution of A x = b        : [1/11, 7/11] = [0.09090909090909091, 0.6363636363636364]

Rosenbrock, variable step        : [0.9999992190902652, 0.9999984335097669] | iterations = 13756
Rosenbrock, nonlinear CG (PR+)   : [1.0000005189805274, 1.0000010380817903] | iterations = 11733
